<a href="https://colab.research.google.com/github/Edubel77/enrutaia/blob/main/notebooks/D1_lab_1_1_entorno.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Lab 1.1 - Paso 1: verificar el entorno
import sys
import numpy as np
import pandas as pd
import tensorflow as tf

print("Python    :", sys.version.split()[0])
print("NumPy     :", np.__version__)
print("Pandas    :", pd.__version__)
print("TensorFlow:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices("GPU"))

Python    : 3.13.18
NumPy     : 2.4.6
Pandas    : 2.2.3
TensorFlow: 2.21.0
GPU disponible: []


In [2]:
# Lab 1.1 - Paso 4: tensores y activaciones

# 1) Crear tensores
escalar = tf.constant(5.0)
vector  = tf.constant([2.0, 1.0, -1.5])
matriz  = tf.constant([[1.0, 2.0],
                       [3.0, 4.0]])

for nombre, t in [("escalar", escalar), ("vector", vector), ("matriz", matriz)]:
    print(f"{nombre:8} -> shape={t.shape}, dtype={t.dtype}")

# 2) Tensor <-> NumPy
arr = matriz.numpy()
print("\nTipo tras .numpy():", type(arr))
print("Matriz @ Matriz (TF):\n", tf.matmul(matriz, matriz).numpy())
print("Matriz @ Matriz (NumPy):\n", arr @ arr)

# 3) Comprobar la tabla de activaciones (z = facturación, soporte, reclamos)
z = vector
print("\nReLU    :", tf.nn.relu(z).numpy())
print("Softmax :", tf.nn.softmax(z).numpy().round(2))
print("Suma softmax:", tf.reduce_sum(tf.nn.softmax(z)).numpy())

escalar  -> shape=(), dtype=<dtype: 'float32'>
vector   -> shape=(3,), dtype=<dtype: 'float32'>
matriz   -> shape=(2, 2), dtype=<dtype: 'float32'>

Tipo tras .numpy(): <class 'numpy.ndarray'>
Matriz @ Matriz (TF):
 [[ 7. 10.]
 [15. 22.]]
Matriz @ Matriz (NumPy):
 [[ 7. 10.]
 [15. 22.]]

ReLU    : [2. 1. 0.]
Softmax : [0.72 0.26 0.02]
Suma softmax: 1.0


In [3]:
# Lab 1.1 - Paso 5: forward pass a mano con NumPy
rng = np.random.default_rng(seed=42)   # semilla fija -> mismos números siempre

def relu(z):
    return np.maximum(0, z)

def softmax(z):
    z = z - z.max(axis=1, keepdims=True)          # truco de estabilidad numérica
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

# Datos: 4 ejemplos, 2 características cada uno
X = np.array([[0.5,  1.0],
              [1.5, -0.5],
              [-1.0, 2.0],
              [0.0,  0.0]])

# Pesos aleatorios (la red aún NO ha aprendido nada)
W1 = rng.normal(0, 1, size=(2, 3));  b1 = np.zeros(3)   # capa oculta
W2 = rng.normal(0, 1, size=(3, 3));  b2 = np.zeros(3)   # capa de salida

# Forward pass
Z1 = X @ W1 + b1
A1 = relu(Z1)
Z2 = A1 @ W2 + b2
Y_pred = softmax(Z2)

print("Shapes -> X:", X.shape, "| W1:", W1.shape, "| A1:", A1.shape, "| W2:", W2.shape, "| Y_pred:", Y_pred.shape)
print("\nProbabilidades por ejemplo (filas) y clase (columnas):\n", Y_pred.round(3))
print("\nSuma por fila:", Y_pred.sum(axis=1).round(6))
print("Clase predicha por ejemplo:", Y_pred.argmax(axis=1))
print("Parámetros totales:", W1.size + b1.size + W2.size + b2.size)

Shapes -> X: (4, 2) | W1: (2, 3) | A1: (4, 3) | W2: (3, 3) | Y_pred: (4, 3)

Probabilidades por ejemplo (filas) y clase (columnas):
 [[0.405 0.249 0.346]
 [0.104 0.684 0.212]
 [0.436 0.217 0.347]
 [0.333 0.333 0.333]]

Suma por fila: [1. 1. 1. 1.]
Clase predicha por ejemplo: [0 1 0 0]
Parámetros totales: 21


In [4]:
# Lab 1.1 - Paso 6: la misma red en Keras
from tensorflow import keras

modelo = keras.Sequential([
    keras.Input(shape=(2,)),                        # 2 características
    keras.layers.Dense(3, activation="relu"),       # capa oculta
    keras.layers.Dense(3, activation="softmax"),    # salida multiclase
])
modelo.summary()

# Copiar TUS pesos de NumPy a Keras
modelo.layers[0].set_weights([W1, b1])
modelo.layers[1].set_weights([W2, b2])

Y_keras = modelo.predict(X, verbose=0)

print("\nKeras:\n", Y_keras.round(3))
print("¿Iguales a tu versión NumPy?", np.allclose(Y_pred, Y_keras, atol=1e-6))

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 3)              │             9 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │            12 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21 (84.00 B)

 Trainable params: 21 (84.00 B)

 Non-trainable params: 0 (0.00 B)


Keras:
 [[0.405 0.249 0.346]
 [0.104 0.684 0.212]
 [0.436 0.217 0.347]
 [0.333 0.333 0.333]]
¿Iguales a tu versión NumPy? True
